<img src="https://raw.githubusercontent.com/stac-utils/stac-fastapi-elasticsearch-opensearch/refs/heads/main/assets/sfeos.png" alt="StacLabs Banner" width="80%">

# Virtual Data Organization & SKOS Semantic Ingestion
### Powering Multi-Tenant GIS Infrastructure via DAG Architecture

This demo showcases the **SFEOS (STAC-FastAPI-OpenSearch)** implementation of the Multi-Tenant Catalogs extension using a real-world GIS use case.

Traditional STAC APIs are restricted to a simple, flat tree structure (one parent per resource). SFEOS implements a **Directed Acyclic Graph (DAG)** architecture, allowing spatial data to be organized into deep, overlapping hierarchies.

This matches how modern urban planning and transportation platforms actually operate: a single spatial asset - like a "School Zone Speed Limit" sign - can belong to a "Regulatory" catalog and a "Warning" catalog simultaneously, without duplicating any underlying data.

**Key Demo Highlights:**
1. **Automated Semantic Ingestion:** Building a deep STAC hierarchy directly from standard SKOS (RDF/XML) vocabularies.
2. **Recursive Discovery:** Navigating nested sub-catalogs across multiple levels of depth.
3. **Poly-hierarchy:** Making a single resource seamlessly discoverable across multiple domain paths with dynamic contextual routing.
4. **Data Safety:** Separating organizational "containers" from the underlying spatial feature data.

### 1. Environment Setup
We begin by installing the necessary SFEOS core components and tools, then downloading and configuring a local OpenSearch instance to act as our high-performance metadata backend.

In [ ]:
!pip install stac-fastapi-opensearch sfeos-tools==0.4.0 requests -q

In [ ]:
import os
import urllib.request
import time

# 1. Configuration
VERSION = "2.11.0"
TAR_FILE = f"opensearch-{VERSION}-linux-x64.tar.gz"
URL = f"https://artifacts.opensearch.org/releases/bundle/opensearch/{VERSION}/{TAR_FILE}"
HOME_DIR = os.getcwd()
OPENSEARCH_DIR = os.path.join(HOME_DIR, f"opensearch-{VERSION}")

print(f"--- Starting Final Robust Setup in {HOME_DIR} ---")

# 2. Download (Skip if exists)
if not os.path.exists(TAR_FILE):
    print(f"Downloading OpenSearch {VERSION}...")
    try:
        urllib.request.urlretrieve(URL, TAR_FILE)
        print("Download complete.")
    except Exception as e:
        print(f"Download failed: {e}")
else:
    print("Archive already exists.")

# 3. Clean up and Extract using System Tar
# This avoids the 'NoSuchFileException' by ensuring all files are extracted
os.system("pkill -f opensearch") # Kill any stuck processes
if os.path.exists(OPENSEARCH_DIR):
    print("Removing corrupted directory...")
    os.system(f"rm -rf {OPENSEARCH_DIR}")

print("Extracting using system 'tar'...")
os.system(f"tar -xzf {TAR_FILE}")
print("✓ Extraction complete.")

# 4. Configure (Overwrite mode 'w')
CONFIG_PATH = os.path.join(OPENSEARCH_DIR, "config/opensearch.yml")
if os.path.exists(CONFIG_PATH):
    print("Applying clean configuration...")
    config_content = (
        "cluster.name: opensearch-cluster\n"
        "node.name: opensearch-node1\n"
        "network.host: 0.0.0.0\n"
        "discovery.type: single-node\n"
        "plugins.security.disabled: true\n"
    )
    with open(CONFIG_PATH, "w") as f:
        f.write(config_content)
    
    # 5. Launch
    print("Launching OpenSearch...")
    os.environ["OPENSEARCH_JAVA_OPTS"] = "-Xms2g -Xmx2g"
    launch_script = os.path.join(OPENSEARCH_DIR, "bin/opensearch")
    
    # Ensure it's executable
    os.system(f"chmod +x {launch_script}")
    
    # Run in background
    os.system(f"nohup {launch_script} > {HOME_DIR}/opensearch_startup.log 2>&1 &")
    
    print("\nSetup complete. Waiting 45s for boot...")
    time.sleep(45)
    
    # 6. Verify
    try:
        response = urllib.request.urlopen("http://localhost:9200", timeout=5)
        print("\nSUCCESS! OpenSearch is responding:")
        print(response.read().decode()[:150] + "...")
    except Exception as e:
        print(f"\nStill failing. Check 'opensearch_startup.log'. Error: {e}")
else:
    print(f"Error: Could not find config at {CONFIG_PATH}")

### 2. Start the SFEOS API
We enable the `ENABLE_CATALOGS_ROUTE` to activate the Multi-Tenant Catalogs extension. 

**Technical Note:** In SFEOS, structural links are **computed at runtime**. This means the API is "self-organizing"—it generates the correct hierarchy links dynamically, ensuring zero metadata "rot" even when the organizational structure is updated.

In [ ]:
import os
import subprocess
import time
import requests

# 1. Configuration for STAC-FastAPI
os.environ["ES_HOST"] = "localhost"
os.environ["ES_PORT"] = "9200"
os.environ["ES_USE_SSL"] = "false"
os.environ["ES_VERIFY_CERTS"] = "false"
os.environ["ENABLE_CATALOGS_ROUTE"] = "true"

def start_stac_api():
    # Check if port 8080 is already in use
    try:
        requests.get("http://localhost:8080", timeout=1)
        print("STAC-FastAPI appears to be already running on port 8080.")
        return
    except requests.exceptions.ConnectionError:
        pass

    # 2. Launch via Subprocess
    print("Starting STAC-FastAPI on port 8080...")

    cmd = [
        "python", "-m", "uvicorn", 
        "stac_fastapi.opensearch.app:app", 
        "--host", "0.0.0.0", 
        "--port", "8080"
    ]

    with open("stac_fastapi.log", "w") as log_file:
        subprocess.Popen(cmd, stdout=log_file, stderr=log_file)

    # 3. Verification Loop
    max_retries = 15
    print("Waiting for API to initialize", end="")
    
    for i in range(max_retries):
        try:
            response = requests.get("http://localhost:8080/", timeout=2)
            if response.status_code == 200:
                print("\n\nSTAC-FastAPI is UP and healthy!")
                return True
        except requests.exceptions.ConnectionError:
            print(".", end="")
            time.sleep(2)
    
    print("\n\nSTAC-FastAPI failed to start in time.")
    print("Check 'stac_fastapi.log' for details.")
    return False

# Execute
start_stac_api()

### 3. Semantic Ingestion (SKOS to STAC)
Transportation authorities and spatial data infrastructures often manage complex asset taxonomies using standard RDF/SKOS vocabularies. SFEOS automatically translates these semantic graphs into a fully navigable STAC Multi-Tenant hierarchy:
* **SKOS Concepts (`skos:Concept`)** → Converted directly into STAC Sub-Catalogs.
* **Hierarchy (`skos:broader` / `skos:narrower`)** → Dynamically mapped to `rel="parent"` and `rel="child"` STAC structural links.
* **Lateral Links (`skos:related` / `skos:exactMatch`)** → Preserved as `rel="related"` metadata links, seamlessly connecting STAC catalogs to external ontologies (like OpenStreetMap tags) or sibling concepts.

In [ ]:
!sfeos-tools ingest-catalog --xml-file traffic-signs.rdf

### 4. Discovery: Exploring the Registry
We query the `/catalogs` endpoint. Notice how SFEOS returns a clean registry of organizational units. Every catalog contains dynamic links that allow clients (like STAC Browser) to crawl the entire DAG without manual configuration.

In [ ]:
import requests
import json

def query_stac(endpoint="/catalogs"):
    """
    Queries the local STAC API and prints formatted JSON output.
    Usage: query_stac("/collections"), query_stac("/"), etc.
    """
    base_url = "http://localhost:8080"
    # Ensure the endpoint starts with a slash
    if not endpoint.startswith("/"):
        endpoint = "/" + endpoint
        
    url = f"{base_url}{endpoint}"
    
    print(f"--- Querying: {url} ---")
    
    try:
        response = requests.get(url, timeout=5)
        
        # Check if the request was successful
        if response.status_code == 200:
            print(json.dumps(response.json(), indent=2))
        else:
            print(f"HTTP Error {response.status_code}: {response.text}")
            
    except requests.exceptions.ConnectionError:
        print("Connection Error: Is the STAC-FastAPI server running on port 8080?")
    except Exception as e:
        print(f"An unexpected error occurred: {e}")

query_stac("/catalogs?limit=3")

### 5. Discovery: Hierarchy Traversal
Large archives require deep nesting. SFEOS supports recursive discovery. Here, we drill down into the sub-catalogs of the **Maximum Speed Limits** topic using the `/catalogs/{id}/catalogs` route.

In [ ]:
query_stac("/catalogs/maximum-speed-limits/catalogs?limit=3")

### 6. Discovery: Thematic Landing Pages
Each catalog acts as a "Virtual Landing Page." Note the `links` array: it provides `rel="child"` links for sub-topics and `rel="parent"` links for breadcrumb navigation.

In [ ]:
query_stac("/catalogs/maximum-speed-limits")

### 7. Virtual Organization: Poly-hierarchy
We will now demonstrate the "Spatial Graph" concept. In this cell, we explore how a single resource (`school-zone-speed`) is linked to **two different catalogs**: "Speed Limits" (Regulatory) and "School Zones" (Warnings). 

This demonstrates **Virtual Organization**: The underlying spatial data is never duplicated or copied. Instead, the API creates overlapping organizational paths to the exact same authoritative STAC collection. This allows different departments - such as traffic enforcement officers and school safety planners - to discover the exact same assets through the thematic context that makes sense to them.

In [ ]:
# 1. Create the collection in 'Speed Limits' (Regulatory)
coll = {
    "id": "school-zone-signs-v1", 
    "type": "Collection", 
    "stac_version": "1.0.0", 
    "description": "Municipal GIS records of active school zone speed limit signs.", 
    "license": "proprietary", 
    "extent": {
        "spatial": {"bbox": [[-180,-90,180,90]]}, 
        "temporal": {"interval": [["2023-01-01T00:00:00Z", None]]}
    }, 
    "links": [], 
    "title": "School Zone Speed Limit Signs"
}
requests.post("http://localhost:8080/catalogs/speed-limit-signs/collections", json=coll)

# 2. Link the SAME collection to 'School Zones' (Warnings)
# Because the ID already exists, the API dynamically maps a new rel="parent" link!
requests.post("http://localhost:8080/catalogs/school-zone-warnings/collections", json=coll)

print("✓ Dataset is now virtually present in BOTH 'Speed Limits' and 'School Zones' catalogs.")

### 8. Discovery: Catalog Collections
This endpoint allows you to filter the spatial archive to see only the datasets (Collections) associated with a specific regulatory or thematic category.

Key architectural observations in the API response:
* **Poly-hierarchy Discovery:** The `rel="related"` links indicate that this Collection also belongs to other parent catalogs (e.g., "School Zones"), providing a complete view of the organizational graph.
* **Authoritative Source:** The `rel="canonical"` link identifies the core global route for the dataset. 

This demonstrates that while the data is discoverable through multiple structural paths, these are **virtual routes**. There is only one authoritative record in the backend database, ensuring data consistency across all departmental "lenses" (e.g., traffic enforcement vs. school safety).

In [ ]:
query_stac("/catalogs/school-zone-warnings/collections")

### 9. Discovery: Scoped Context
When accessing a collection through a specific catalog, SFEOS dynamically locks the `rel="parent"` link to that exact catalog context. This ensures that if a GIS analyst navigates to the dataset via the "School Zones" catalog, the UI (like STAC Browser) keeps their breadcrumb trail locked within the "School Zones" path, rather than unexpectedly redirecting them to "Speed Limits".

In [ ]:
query_stac("/catalogs/school-zone-warnings/collections/school-zone-signs-v1")

### 10. Operational Management and Data Safety
Catalog management in SFEOS is strictly **non-destructive**. Deleting a catalog container simply removes that specific organizational view, but it has zero impact on the underlying spatial data. 

In this step, we will delete the "School Zones" catalog. Even if our `school-zone-signs-v1` collection were removed from **all** virtual catalogs, it would still persist safely in the core STAC API under the global `/collections` route. This architecture ensures that departmental restructuring or taxonomy updates can never result in accidental data loss or "orphaned" GIS assets.

In [ ]:
# Delete an organizational container
print("Deleting organizational container: School Zones...")
requests.delete("http://localhost:8080/catalogs/school-zone-warnings")

# Verify data persistence at the global level
r = requests.get("http://localhost:8080/collections/school-zone-signs-v1")
if r.status_code == 200:
    print("✓ Data Safety Confirmed: The 'School Zones' organizational view was removed, but the spatial data is preserved.")

## Conclusion: A Modern Approach to Spatial Data Governance

This demo illustrates how the SFEOS Multi-Tenant architecture moves beyond simple data listing and becomes a **dynamic organizational layer** for enterprise GIS and spatial archives. 

For large-scale municipalities, transportation authorities, and infrastructure partners, the benefits are clear:

### Flexible Data Discovery (The "Asset Type vs. Jurisdiction" Problem)
Traditionally, spatial data is forced into a rigid, single-path folder structure (e.g., organized strictly by Asset Type, like "Traffic Signs"). With SFEOS, you can provide **Asset-specific views** AND **Thematic/Jurisdictional views** (like "School Zones" or "Downtown District") simultaneously. You don't need to move or duplicate data - you simply create a new "Virtual Catalog" that points to the existing collections.

### Semantic Alignment via SKOS
By supporting **automated SKOS ingestion**, the API remains technically aligned with official data standards. You can ingest an official municipal taxonomy or national transport vocabulary (like INSPIRE) and instantly generate a STAC hierarchy that matches it. This bridges the gap between urban planning ontologies and technical data access.

### Intelligent Navigation
The system handles the "plumbing" of metadata links automatically. Because links are **context-aware**, an analyst who enters through a "School Zones" catalog will see "School Zones" as the parent of the data they are viewing. This creates a seamless, intuitive experience for users exploring the archive via tools like the STAC Browser.

### Non-Destructive Management
Data safety is built into the architecture. You can create, reorganize, or retire catalogs as departmental needs change without any risk to the underlying spatial data. Because collections are ultimately anchored in the core `/collections` route, deleting an organizational container cannot result in data loss. If an asset loses its last parent catalog, it remains safely discoverable at the Root level.

### Operational Efficiency
Because all structural relationships are computed at runtime via the DAG architecture, there is **zero overhead** for re-indexing when links change. This allows enterprise GIS teams to manage millions of spatial features and thousands of virtual catalogs with high performance and incredibly low administrative maintenance.

---
**SFEOS provides the tools to build a truly interconnected spatial data ecosystem that is as easy to manage as it is to explore.**